In [1]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False 
matplotlib.rcParams['font.family'] = 'Malgun Gothic' 
matplotlib.rcParams['font.size'] 
import pandas as pd
import numpy as np

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
import requests
from bs4 import BeautifulSoup
import time
import os
from dotenv import load_dotenv

In [ ]:
# url = "https://riahn.co.kr/niabbs5/inc.php?inc=sub4"
# options = Options()
# options.add_experimental_option("excludeSwitches", ["enable-automation"])
# options.add_experimental_option("useAutomationExtension", False)
# options.add_argument("--disable-blink-features=AutomationControlled")
# browser = webdriver.Chrome(options=options)
# browser.maximize_window() 
# browser.get(url)
# time.sleep(2)

# soup = BeautifulSoup(browser.page_source,'lxml')
# with open('file/riahn.htnl','w',encoding='utf-8') as f:
#     f.write(soup.prettify())

In [4]:
soup

<html class="hydrated"><head><style data-styles="">ion-icon{visibility:hidden}.hydrated{visibility:inherit}</style>
<!--260624  -->
<meta content="8CBxssCvQ_CqSDBbFa0h6EduKS_JANseiwIMn6np8pA" name="google-site-verification"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=Edge,chrome=1" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<script async="" src="https://cdn.channel.io/plugin/ch-plugin-web.js" type="text/javascript"></script><script async="" src="https://connect.facebook.net/en_US/fbevents.js"></script><script src="https://ajax.googleapis.com/ajax/libs/jquery/1.4/jquery.min.js" type="text/javascript"></script>
<script src="js/slimbox2.js" type="text/javascript"></script><script src="https://www.phonetalk.kr/esti/ads.js"></script>
<link href="css_2022/slimbox2.css" media="screen" rel="stylesheet" type="text/css"/>
<link href="js/css/lightbox.css" media="screen" rel="stylesheet" type="

In [7]:
url = "https://riahn.co.kr/niabbs5/inc.php?inc=sub4"

options = Options()
options.add_experimental_option(
    "excludeSwitches", ["enable-automation"]
)
options.add_experimental_option(
    "useAutomationExtension", False
)
options.add_argument(
    "--disable-blink-features=AutomationControlled"
)
options.add_experimental_option(
    "prefs",
    {"profile.managed_default_content_settings.images": 2}
)

browser = webdriver.Chrome(options=options)
browser.maximize_window()
browser.get(url)
time.sleep(2)

# 5. 자동 스크롤
previous_height = 0
same_height = 0

for _ in range(30):
    browser.execute_script("window.scrollBy(0, 700)")
    time.sleep(0.5)

    current_height = browser.execute_script(
        "return document.documentElement.scrollHeight"
    )

    if current_height == previous_height:
        same_height += 1
    else:
        same_height = 0

    previous_height = current_height

    if same_height >= 3:
        break

# 6. HTML 분석
soup = BeautifulSoup(browser.page_source, "lxml")

# 7. HTML 파일 저장
os.makedirs("file", exist_ok=True)

with open("file/riahn.html", "w", encoding="utf-8") as f:
    f.write(soup.prettify())

# 8. 기존 클래스 확인
datas = soup.find("div", class_="subtop")

if datas is not None:
    s_datas = datas.find("div", class_="subtop3_t")
else:
    s_datas = None

print("subtop:", datas is not None)
print("subtop3_t:", s_datas is not None)

# 9. 가맹조건 표 스크래핑
print("\n===== 가맹조건 =====")

for table in soup.find_all("table"):
    table_text = table.get_text(" ", strip=True)

    if any(word in table_text for word in ["보증금", "가맹비"]):
        for tr in table.find_all("tr"):
            values = [
                cell.get_text(" ", strip=True)
                for cell in tr.find_all(["th", "td"])
            ]

            if values:
                print(values)

# 10. 가맹조건 및 개설절차 관련 HTML 요소 검색
print("\n===== 가맹조건 및 개설절차 =====")

found = set()

for tag in soup.find_all(True):
    text = tag.get_text(" ", strip=True)

    if (
        ("가맹 조건" in text or "개설절차" in text)
        and len(text) <= 100
        and text not in found
    ):
        found.add(text)
        print("태그:", tag.name)
        print("클래스:", tag.get("class"))
        print("내용:", text)
        print("-" * 40)

# 11. 실제 HTML 구조 확인
print("\nHTML 저장 완료: file/riahn.html")

# 브라우저 종료
browser.quit()

subtop: True
subtop3_t: False

===== 가맹조건 =====
['구분', '내용']
['보증금', '5,000,000원 - 계약해지시 반환 - 보증보험증권 발행가능 - 보증보험 대체 - 수수료 약 30여만원(3년기준)']
['가맹비', '11,000,000(VAT포함)']
['총합계', '16,000,000']
['월 로열티', '40평 미만 월 600,000원(VAT별도) 40평 이상 월 800,000원(VAT별도)']
['인테리어', '※ 인테리어업체 선정은 가맹점주 선택']
['간판', '일반 플렉스 간판 - 1회배당×150,000원(VAT별도) 입체형 간판(LED 간판) - 1자×170,000원(VAT별도)']

===== 가맹조건 및 개설절차 =====
태그: div
클래스: ['subtop2', 'aos-init', 'aos-animate']
내용: 가맹 조건 & 개설절차
----------------------------------------

HTML 저장 완료: file/riahn.html


In [ ]:
riahn_data_1=[]
for div in divs :
    title_tag = div.find('a')
    title = title_tag.get_text(strip=True) if title_tag else None
    # print(title_tag)
    first_url = 'https://namu.wiki'
    url = first_url+title_tag['href']
    # print(url)
    namu_data_1.append({
        'title': title,
        'url': url
    })